In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
import numpy as np
import gc
from tqdm import tqdm

from unet_model import LucidRF_UNet
from unet_dataset import LucidRFDataset
from features import calc_sinr_db
from earlystopping import *

In [ ]:
BATCH_SIZE = 4
NUM_WORKERS = 0
PYTORCH_THREADS = 8
LEARNING_RATE = 1e-4  # Standard starting point for U-Nets
NUM_EPOCHS = 20
VAL_SPLIT = 0.2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(PYTORCH_THREADS)

print(f"Device: {DEVICE}")

print(f"Threads capped at: {PYTORCH_THREADS}")
print(f"Batch Size: {BATCH_SIZE}")

In [ ]:
print("\n[1/4] Loading Datasets...")

spot_ds = LucidRFDataset(MIT_SPOT_X_NORMALIZED, MIT_SPOT_Y_NORMALIZED)
barrage_ds = LucidRFDataset(MIT_BARRAGE_X_NORMALIZED, MIT_BARRAGE_Y_NORMALIZED)

print(f"Loaded Spot Data: {len(spot_ds)} samples")
print(f"Loaded Barrage Data: {len(barrage_ds)} samples")

full_dataset = ConcatDataset([spot_ds, barrage_ds])
total_samples = len(full_dataset)
print(f"Total Training Pool: {total_samples}")


val_size = int(total_samples * VAL_SPLIT)
train_size = total_samples - val_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
print("\n[2/4] Initializing U-Net...")
model = LucidRF_UNet(n_channels=2, n_classes=2, base=16).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

# Initialize Callbacks
early_stopper = EarlyStopping(patience=5, min_delta=1e-6)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
start_epoch = 0
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_sinr': [], 'val_sinr': []}

if os.path.exists(MACHINE_B_MODEL_FILE):
    print(f"Found checkpoint: {MACHINE_B_MODEL_FILE}")
    try:
        checkpoint = torch.load(MACHINE_B_MODEL_FILE, map_location=DEVICE)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch']
        best_val_loss = checkpoint['best_loss']
        
        print(f"Resuming from End of Epoch {start_epoch} | Best Loss: {best_val_loss:.6f}")
            
    except Exception as e:
        print(f"Critical Error loading checkpoint: {e}")
        print("If the file is old/incompatible, delete it manually and restart.")
else:
    print("No checkpoint found. Starting fresh.")

print("\n[3/4] Starting Training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    running_sinr = 0.0
    
    # Progress bar for training
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for inputs, targets in pbar:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        sinr = calculate_sinr_db(outputs, targets)
        
        running_loss += loss.item()
        running_sinr += sinr

        pbar.set_postfix({'loss': f"{loss.item():.5f}", 'sinr': f"{sinr:.1f} dB"})
    
    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_sinr = running_sinr / len(train_loader)
    
    # Validation Phase
    model.eval()
    val_running_loss = 0.0
    val_running_sinr = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)

            loss = criterion(outputs, targets)
            sinr = calculate_sinr_db(outputs, targets)
            
            val_running_loss += loss.item()
            val_running_sinr += sinr
            
    epoch_val_loss = val_running_loss / len(val_loader)
    epoch_val_sinr = val_running_sinr / len(val_loader)
    
    # Store history
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_sinr'].append(epoch_train_sinr)
    history['val_sinr'].append(epoch_val_sinr)
    
    print(f"   Shape Check: In {inputs.shape} -> Out {outputs.shape}") # Sanity check once per epoch
    print(f"   Results: Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f}")


    # Checkpointing
    if epoch_val_loss < best_val_loss:
        print(f"Improvement! Saving model... ({best_val_loss:.6f} -> {epoch_val_loss:.6f})")
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), MACHINE_B_MODEL_FILE)

print("\n[4/4] Training Complete.")